# Data Modelling and Clustering

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('cleaned_house_data.csv')
print(df.info())
print(df.head())

## 1. Geographical Distribution of Cleaned House Data

In [ ]:
# Setup & install folium if needed
import sys
try:
    import folium
    from folium.plugins import MarkerCluster
except ImportError:
    print("Installing folium...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])
    import folium
    from folium.plugins import MarkerCluster

import os
import kagglehub
import pandas as pd
import numpy as np

# Download original data to fetch lat and long
path = kagglehub.dataset_download("harlfoxem/housesalesprediction")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df_orig = pd.read_csv(os.path.join(path, csv_file))

# Clean it using the same steps to align row indices
df_orig_cleaned = df_orig[df_orig["bathrooms"] > 0].reset_index(drop=True)
df_orig_cleaned = df_orig_cleaned[df_orig_cleaned["bedrooms"] > 0].reset_index(drop=True)
df_orig_cleaned.loc[df_orig_cleaned["bedrooms"] == 33, "bedrooms"] = 3
df_orig_cleaned = (df_orig_cleaned.sort_values("date")
                  .drop_duplicates(subset="id", keep="last")
                  .reset_index(drop=True))

# Create a mapping dataframe with original attributes and coordinates
df_map = df.copy()
df_map['lat'] = df_orig_cleaned['lat']
df_map['long'] = df_orig_cleaned['long']
df_map['price'] = df_orig_cleaned['price']
df_map['zipcode'] = df_orig_cleaned['zipcode']
df_map['bedrooms_orig'] = df_orig_cleaned['bedrooms']
df_map['bathrooms_orig'] = df_orig_cleaned['bathrooms']

print("Geographical dataframe prepared. Shape:", df_map.shape)

In [ ]:
# Display geographical distribution of house prices
def display_cleaned_data_map():
    print(f"Plotting all cleaned house data distribution ({len(df_map)} properties total)...")
    
    # No sampling, plot all properties
    plot_data = df_map
        
    seattle_lat, seattle_long = 47.6062, -122.3321
    # Enable standard zoom controls and scroll/touch gestures
    m = folium.Map(location=[seattle_lat, seattle_long], zoom_start=11, tiles="cartodbpositron")
    
    for idx, row in plot_data.iterrows():
        price_val = row['price']
        price_str = f"${price_val:,.0f}"
        
        html_popup = f'''
        <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4;">
            <h4 style="margin: 0 0 5px 0; color: #10B981;">Property Details</h4>
            <b>Price:</b> {price_str}<br>
            <b>Zipcode:</b> {int(row['zipcode'])}<br>
            <b>Bedrooms:</b> {int(row['bedrooms_orig'])}<br>
            <b>Bathrooms:</b> {row['bathrooms_orig']}<br>
            <b>Luxury Score:</b> {row['luxury_score']:.2f}<br>
        </div>
        '''
        
        # Color based on price
        if price_val < 350000:
            color = "#10B981"  # Emerald green for affordable
        elif price_val < 650000:
            color = "#3B82F6"  # Blue for mid-range
        else:
            color = "#EF4444"  # Red for high-end
            
        # Add markers directly to map with small radius for overlay visualization
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=2.0,
            weight=0,
            popup=folium.Popup(html_popup, max_width=200),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6
        ).add_to(m)
        
    return m

m_cleaned = display_cleaned_data_map()
m_cleaned

## 2. Elbow Method to Determine Optimal Clusters

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

X = df.values
sse = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    sse.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), sse, marker='o', linestyle='--', color='b')
plt.xlabel("Number of Clusters K")
plt.ylabel("SSE (Inertia)")
plt.title("Elbow Method for K Selection")
plt.grid(True)
plt.savefig('elbow_curve.png', dpi=300)
plt.show()

## 3. Comparing Clustering Algorithms (K-Means, GMM, Agglomerative, DBSCAN, HDBSCAN)

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import time

results = []

# 1. K-Means
t0 = time.time()
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X)
t1 = time.time()
results.append({
    'Algorithm': 'K-Means',
    'Silhouette': silhouette_score(X, labels_kmeans),
    'Davies-Bouldin': davies_bouldin_score(X, labels_kmeans),
    'Time (s)': t1 - t0
})

# 2. Gaussian Mixture Model
t0 = time.time()
gmm = GaussianMixture(n_components=4, random_state=42, n_init=5)
labels_gmm = gmm.fit_predict(X)
t1 = time.time()
results.append({
    'Algorithm': 'Gaussian Mixture (GMM)',
    'Silhouette': silhouette_score(X, labels_gmm),
    'Davies-Bouldin': davies_bouldin_score(X, labels_gmm),
    'Time (s)': t1 - t0
})

# 3. Agglomerative Clustering (Hierarchical)
t0 = time.time()
agglo = AgglomerativeClustering(n_clusters=4)
labels_agglo = agglo.fit_predict(X)
t1 = time.time()
results.append({
    'Algorithm': 'Agglomerative (Full)',
    'Silhouette': silhouette_score(X, labels_agglo),
    'Davies-Bouldin': davies_bouldin_score(X, labels_agglo),
    'Time (s)': t1 - t0
})

# 4. DBSCAN (excluding noise label -1 for metrics evaluation)
t0 = time.time()
dbscan = DBSCAN(eps=1.0, min_samples=10)
labels_db = dbscan.fit_predict(X)
t1 = time.time()

mask_db = labels_db != -1
if np.sum(mask_db) > 0 and len(np.unique(labels_db[mask_db])) > 1:
    db_silhouette = silhouette_score(X[mask_db], labels_db[mask_db])
    db_db_score = davies_bouldin_score(X[mask_db], labels_db[mask_db])
else:
    db_silhouette = np.nan
    db_db_score = np.nan

results.append({
    'Algorithm': 'DBSCAN (Excluding Noise)',
    'Silhouette': db_silhouette,
    'Davies-Bouldin': db_db_score,
    'Time (s)': t1 - t0
})

# 5. HDBSCAN (excluding noise label -1 for metrics evaluation)
t0 = time.time()
hdb = HDBSCAN(min_cluster_size=100)
labels_hdb = hdb.fit_predict(X)
t1 = time.time()

mask_hdb = labels_hdb != -1
if np.sum(mask_hdb) > 0 and len(np.unique(labels_hdb[mask_hdb])) > 1:
    hdb_silhouette = silhouette_score(X[mask_hdb], labels_hdb[mask_hdb])
    hdb_db_score = davies_bouldin_score(X[mask_hdb], labels_hdb[mask_hdb])
else:
    hdb_silhouette = np.nan
    hdb_db_score = np.nan

results.append({
    'Algorithm': 'HDBSCAN (Excluding Noise)',
    'Silhouette': hdb_silhouette,
    'Davies-Bouldin': hdb_db_score,
    'Time (s)': t1 - t0
})

res_df = pd.DataFrame(results)
print(res_df.to_string())

# Save labels for all models to df
df['Cluster_KMeans'] = labels_kmeans
df['Cluster_GMM'] = labels_gmm
df['Cluster_DBSCAN'] = labels_db
df['Cluster_Agglo'] = labels_agglo
df['Cluster_HDBSCAN'] = labels_hdb
df.to_csv('clustered_data.csv', index=False)

## 4. Profiling Clusters (Feature Averages)

In [ ]:
# Define a helper function to drop label columns before averaging
def get_clean_profile(cluster_df):
    cols_to_drop = [c for c in cluster_df.columns if c.startswith('Cluster_') or c.startswith('PCA') or c.startswith('TSNE')]
    return cluster_df.drop(columns=cols_to_drop).mean()

# 1. K-Means
print("=== K-Means Feature Averages for Each Cluster ===")
for i in range(4):
    cluster_data = df[df['Cluster_KMeans'] == i]
    print(f"\nCluster {i} (Sample Count: {len(cluster_data)}):")
    print(get_clean_profile(cluster_data))
print("===================================================\n")

# 2. GMM
print("=== GMM Feature Averages for Each Cluster ===")
for i in range(4):
    cluster_data = df[df['Cluster_GMM'] == i]
    print(f"\nCluster {i} (Sample Count: {len(cluster_data)}):")
    print(get_clean_profile(cluster_data))
print("===================================================\n")

# 3. Agglomerative (Hierarchical)
print("=== Agglomerative Feature Averages for Each Cluster ===")
for i in range(4):
    cluster_data = df[df['Cluster_Agglo'] == i]
    print(f"\nCluster {i} (Sample Count: {len(cluster_data)}):")
    print(get_clean_profile(cluster_data))
print("===================================================\n")

# 4. DBSCAN
unique_db_labels = [l for l in np.unique(labels_db) if l != -1]
print(f"=== DBSCAN Feature Averages per Cluster (Clusters found: {len(unique_db_labels)}) ===")
for label in unique_db_labels:
    cluster_data = df[df['Cluster_DBSCAN'] == label]
    print(f"\nCluster {label} (Sample Count: {len(cluster_data)}):")
    print(get_clean_profile(cluster_data))
noise_count_db = np.sum(labels_db == -1)
print(f"\nNoise points detected by DBSCAN: {noise_count_db}/{len(df)}")
print("===================================================\n")

# 5. HDBSCAN
unique_hdb_labels = [l for l in np.unique(labels_hdb) if l != -1]
print(f"=== HDBSCAN Feature Averages per Cluster (Clusters found: {len(unique_hdb_labels)}) ===")
for label in unique_hdb_labels:
    cluster_data = df[df['Cluster_HDBSCAN'] == label]
    print(f"\nCluster {label} (Sample Count: {len(cluster_data)}):")
    print(get_clean_profile(cluster_data))
noise_count_hdb = np.sum(labels_hdb == -1)
print(f"\nNoise points detected by HDBSCAN: {noise_count_hdb}/{len(df)}")
print("===================================================\n")

## 5. UMAP Presence Verification

In [ ]:
try:
    import umap
    print("UMAP is installed")
except:
    print("UMAP not installed")

## 6. Visualizing Clusters (PCA & t-SNE Projections per Model)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 1. Compute Projections
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

np.random.seed(42)
sample_idx = np.random.choice(len(df), 5000, replace=False)
X_tsne_sample = X[sample_idx]
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_tsne_sample)

df_tsne = df.iloc[sample_idx].copy()
df_tsne['TSNE1'] = X_tsne[:, 0]
df_tsne['TSNE2'] = X_tsne[:, 1]

# 2. Define a helper function to plot and save graphics for each model
def plot_and_save_model(model_name, label_col):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # PCA Plot
    sns.scatterplot(x='PCA1', y='PCA2', hue=label_col, palette='tab10', data=df, ax=axes[0], s=15, alpha=0.6)
    axes[0].set_title(f'{model_name} Clustering (PCA Projection)')
    
    # t-SNE Plot
    sns.scatterplot(x='TSNE1', y='TSNE2', hue=label_col, palette='tab10', data=df_tsne, ax=axes[1], s=15, alpha=0.7)
    axes[1].set_title(f'{model_name} Clustering (t-SNE Projection)')
    
    plt.tight_layout()
    filename = f"{model_name.lower().replace(' ', '_')}_clusters.png"
    plt.savefig(filename, dpi=300)
    plt.show()
    print(f"Saved plot for {model_name} as {filename}")

# Generate and save graphics for all five models
plot_and_save_model('K-Means', 'Cluster_KMeans')
plot_and_save_model('GMM', 'Cluster_GMM')
plot_and_save_model('Agglomerative', 'Cluster_Agglo')
plot_and_save_model('DBSCAN', 'Cluster_DBSCAN')
plot_and_save_model('HDBSCAN', 'Cluster_HDBSCAN')

## 8. Geographical Visualization of Clusters

In [ ]:
# 2. Interactive Map using Folium
import sys

# Auto-install folium if missing
try:
    import folium
except ImportError:
    print("Installing folium...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])
    import folium

def display_all_clusters_map(model_name):
    """
    Plots all properties on a Seattle map, colored by their cluster label.
    """
    model_col = f"Cluster_{model_name.replace('-', '')}"
    if model_col not in df.columns:
        print(f"Error: Column {model_col} not found. Please run clustering first.")
        return
        
    print(f"Plotting all clusters for {model_name} ({len(df)} properties total)...")
    
    # Plot all properties (no sampling)
    plot_data = df.copy()
        
    # Align coordinates using our preloaded df_map
    plot_data['lat'] = df_map.loc[plot_data.index, 'lat']
    plot_data['long'] = df_map.loc[plot_data.index, 'long']
    plot_data['price'] = df_map.loc[plot_data.index, 'price']
    plot_data['zipcode'] = df_map.loc[plot_data.index, 'zipcode']
    plot_data['bedrooms_orig'] = df_map.loc[plot_data.index, 'bedrooms_orig']
    plot_data['bathrooms_orig'] = df_map.loc[plot_data.index, 'bathrooms_orig']
    
    # Center map on Seattle, enable all standard zoom functions
    seattle_lat, seattle_long = 47.6062, -122.3321
    m = folium.Map(location=[seattle_lat, seattle_long], zoom_start=11, tiles="cartodbpositron")
    
    # Palette for up to 10 clusters (distinct hex colors)
    palette = ['#3B82F6', '#F59E0B', '#10B981', '#EF4444', '#4F46E5', '#8B5CF6', '#EC4899', '#14B8A6', '#06B6D4', '#84CC16']
    
    for idx, row in plot_data.iterrows():
        price_val = row['price']
        price_str = f"${price_val:,.0f}"
        cluster_val = int(row[model_col])
        
        # Determine color and label
        if cluster_val == -1:
            color = "#94A3B8"  # Slate grey for noise
            cluster_name = "Noise (Outlier)"
        else:
            color = palette[cluster_val % len(palette)]
            cluster_name = f"Cluster {cluster_val}"
            
        html_popup = f'''
        <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4;">
            <h4 style="margin: 0 0 5px 0; color: {color};">{cluster_name}</h4>
            <b>Price:</b> {price_str}<br>
            <b>Zipcode:</b> {int(row['zipcode'])}<br>
            <b>Bedrooms:</b> {int(row['bedrooms_orig'])}<br>
            <b>Bathrooms:</b> {row['bathrooms_orig']}<br>
            <b>Luxury Score:</b> {row['luxury_score']:.2f}<br>
        </div>
        '''
        
        # Add CircleMarker directly to map without MarkerCluster
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=2.0,
            weight=0,
            popup=folium.Popup(html_popup, max_width=200),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6
        ).add_to(m)
        
    # Add a simple HTML legend
    legend_html = f'''
    <div style="
        position: fixed; 
        bottom: 50px; left: 50px; width: 150px; height: auto; 
        background-color: white; border:2px solid grey; z-index:9999; 
        font-family: Arial; font-size:12px; padding: 10px;
        border-radius: 6px; box-shadow: 2px 2px 5px rgba(0,0,0,0.2);
    ">
    <h4 style="margin: 0 0 8px 0; font-size: 13px; color: #374151;">{model_name} Legend</h4>
    '''
    
    unique_clusters = sorted(plot_data[model_col].unique())
    for c in unique_clusters:
        c = int(c)
        if c == -1:
            color = "#94A3B8"
            name = "Noise"
        else:
            color = palette[c % len(palette)]
            name = f"Cluster {c}"
        legend_html += f'<div style="margin-bottom: 4px;"><span style="display:inline-block; width:12px; height:12px; background-color:{color}; border-radius:50%; margin-right:8px; vertical-align:middle;"></span>{name}</div>'
    
    legend_html += '</div>'
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

# Choose preferred model to visualize (available: 'KMeans', 'GMM', 'DBSCAN', 'Agglo', 'HDBSCAN')
preferred_model = 'KMeans'

m = display_all_clusters_map(preferred_model)
m